# B1.2 · Context engineering for code review

**Function B — Product & Application Security → The AppSec Engineer / Code Reviewer**  ·  *AI for Security*

---

**Risk.** Context stuffing blows the budget and buries the sink.

**Control.** Repo maps, prompt caching, tool-mediated retrieval over stuffing.

**This lab.** Beat context stuffing with tool-mediated retrieval.

| | |
|---|---|
| Open-source tooling | ripgrep, tree-sitter |
| Open-weight models | GLM-4.6 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B1.2"))

Context engineering for code review is mostly subtraction. The model does not need the repository; it needs the sink, the source, and the path between them.

In [ ]:
from cybercommons import appsec

src = appsec.SNIPPETS["sql_injection"]
finding = appsec.scan("sql_injection", src)[0]

def context_whole_file(source, _f):
    return source

def context_windowed(source, f, radius=2):
    lines = source.splitlines()
    lo, hi = max(f.line - radius - 1, 0), min(f.line + radius, len(lines))
    return "\n".join(lines[lo:hi])

for name, fn in (("whole file", context_whole_file), ("windowed", context_windowed)):
    ctx = fn(src, finding)
    print(f"--- {name}: {len(ctx)} chars, {len(ctx.split())} tokens-ish")
    print(ctx)
    print()

The windowed context is a fraction of the size and contains the entire finding. Scale that across a repository and the difference is not cost — it is whether the relevant line survives the context window at all.

### Expect

The windowed context is several times smaller than the whole file and still contains the concatenated query on the finding's line.

### Your turn

Windowing loses the definition of the caller, which is where reachability is decided. Add just the enclosing function signature and measure the size cost. That trade is context engineering.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B1.2.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*